In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
enron = pd.read_csv('data/enron_email_context/EnronEmailReplyPairsWithContext.csv')

In [4]:
enron.head(10)

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,6/18/2001 22:44,6/19/2001 15:49,NaN
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,3/19/2002 8:30,3/19/2002 8:34,NaN
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,1/15/2002 12:30,1/15/2002 16:29,NaN
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,1/19/2002 10:55,1/19/2002 10:56,NaN
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,1/21/2002 15:30,1/21/2002 15:44,NaN
5,What are they having?,Need you or John to step out. : Louise.Kitchen...,RE:,Re: Did you sneak out?,louise.kitchen@enron.com,8777865122@skytel.com,1/21/2002 16:26,1/23/2002 9:24,NaN
6,What is the situation with a lawsuit involving...,Solarc is the system used for NGLs etc - we do...,Solarc,RE: Solarc,8777865122@skytel.com,louise.kitchen@enron.com,9/1/2002 8:23,9/1/2002 8:27,NaN
7,Did you get tg back to 1yr docs,Did not get a commitment. Just told them I tho...,Re: Solarc,Re: Solarc,louise.kitchen@enron.com,8777865122@skytel.com,10/1/2002 10:32,10/1/2002 11:00,NaN
8,What do you think happens if they require 2yrs?,50% or less of the current group. We have a ma...,Contracts,RE: Contracts,8777865122@skytel.com,louise.kitchen@enron.com,10/1/2002 11:10,10/1/2002 11:15,NaN
9,We are going to have to work on making this mo...,"I understand. If I have too, I will dip into m...",Money,RE: Money,8777865122@skytel.com,m..presto@enron.com,1/19/2002 10:31,1/19/2002 15:49,NaN


In [7]:
# Use Regex to parse out any unwanted characters
enron['DateSend'] = enron['DateSend'].astype(str).str.replace(r'[^\w\s,:/:-]', '', regex=True).str.strip()
enron['DateReply'] = enron['DateReply'].astype(str).str.replace(r'[^\w\s,:/:-]', '', regex=True).str.strip()

# Convert date columns to datetime objects and fill w NaN values
enron['DateSend'] = pd.to_datetime(enron['DateSend'], format='mixed', errors='coerce')
enron['DateReply'] = pd.to_datetime(enron['DateReply'], format='mixed', errors='coerce')

# Calculate if reply was within 30 minutes and convert True/False to 1/0
enron['Response_Within_30_Mins'] = (
    (enron['DateReply'] - enron['DateSend']) <= pd.Timedelta(minutes=30)
).astype(int)

# Check the breakdown of 1s and 0s
enron['Response_Within_30_Mins'].value_counts()

Response_Within_30_Mins
0    8416
1    6961
Name: count, dtype: int64

In [9]:
# Extract the hour of the day (0-23) - people don't reply fast at 3 AM!
enron['HourSent'] = enron['DateSend'].dt.hour

# Extract day of the week (0=Monday, 6=Sunday)
enron['DaySent'] = enron['DateSend'].dt.dayofweek

# Create a binary flag for weekends
enron['IsWeekend'] = enron['DaySent'].isin([5, 6]).astype(int)

In [26]:
enron.head()

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response_Within_30_Mins,HourSent,DaySent,IsWeekend
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,0,22.0,0.0,0
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,1,8.0,1.0,0
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,0,12.0,1.0,0
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,1,10.0,5.0,1
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,1,15.0,0.0,0


In [12]:
enron['EmailReply'].isna().sum()

np.int64(0)

In [13]:
emails = pd.read_csv('data/emails.csv')

In [28]:
emails['file'][2]

'allen-p/_sent_mail/100.'

In [22]:
emails['message'][2]

"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Leah Van Arsdall\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Dec2000\\Notes Folders\\'sent mail\nX-Origin: Allen-P\nX-FileName: pallen.nsf\n\ntest successful.  way to go!!!"

In [30]:
from email import message_from_string

# Pull out From, To, Subject, Date, Body from each raw message
def parse_email(raw):
    msg = message_from_string(raw)
    return pd.Series({
        'From': msg.get('From'),
        'To': msg.get('To'),
        'Subject': msg.get('Subject', ''),
        'Date': msg.get('Date'),
        'Body': msg.get_payload()
    })

parsed = emails['message'].apply(parse_email)
emails = pd.concat([emails, parsed], axis=1)
emails['Date'] = pd.to_datetime(emails['Date'], format='mixed', errors='coerce', utc=True)

emails['Subject'] = emails['Subject'].fillna('')
emails['is_reply'] = emails['Subject'].str.lower().str.startswith('re:')

# Thread key: ignores Re: and ignores direction, so a question and
# its answer land on the same key
emails['clean_subject'] = emails['Subject'].str.replace('re: ', '', case=False).str.strip().str.lower()
emails['pair_key'] = emails.apply(
    lambda r: tuple(sorted([str(r['From']).lower(), str(r['To']).lower()])) + (r['clean_subject'],),
    axis=1
)

# Which thread keys have at least one reply in them?
keys_with_reply = set(emails.loc[emails['is_reply'], 'pair_key'])

# Mark every original (non-reply) email: did its thread ever get a reply?
emails['Response'] = 0
emails.loc[~emails['is_reply'], 'Response'] = emails.loc[~emails['is_reply'], 'pair_key'].isin(keys_with_reply).astype(int)

# Keep only originals with NO response
no_response = emails[(emails['is_reply'] == False) & (emails['Response'] == 0)].copy()

result = pd.DataFrame({
    'EmailSend': no_response['Body'],
    'EmailReply': np.nan,
    'SubjectSend': no_response['Subject'],
    'SubjectReply': np.nan,
    'From': no_response['From'],
    'To': no_response['To'],
    'DateSend': no_response['Date'],
    'DateReply': pd.NaT,
    'Context': np.nan,
    'Response': 0   # new indicator column: 1 = got a reply, 0 = no reply
})

result.to_csv('data/emails_no_response_full_schema.csv', index=False)
print(f"No-response emails: {len(result)}")
print(result.head())

No-response emails: 344374
                                            EmailSend  EmailReply  \
0                           Here is our forecast\n\n          NaN   
3   Randy,\n\n Can you send me a schedule of the s...         NaN   
6   Please cc the following distribution list with...         NaN   
9   ---------------------- Forwarded by Phillip K ...         NaN   
11  Lucy,\n\n Here are the rentrolls:\n\n\n\n Open...         NaN   

                                          SubjectSend  SubjectReply  \
0                                                               NaN   
3                                                               NaN   
6                                                               NaN   
9   FW: fixed forward or other Collar floor gas pr...           NaN   
11                                                              NaN   

                       From                                                To  \
0   phillip.allen@enron.com                       

In [35]:
result.head()

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response
0,Here is our forecast\n\n,NaN,,NaN,phillip.allen@enron.com,tim.belden@enron.com,2001-05-14 23:39:00+00:00,NaT,NaN,0
3,"Randy,\n\n Can you send me a schedule of the s...",NaN,,NaN,phillip.allen@enron.com,randall.gay@enron.com,2000-10-23 13:13:00+00:00,NaT,NaN,0
6,Please cc the following distribution list with...,NaN,,NaN,phillip.allen@enron.com,"david.l.johnson@enron.com, john.shafer@enron.com",2000-08-22 14:44:00+00:00,NaT,NaN,0
9,---------------------- Forwarded by Phillip K ...,NaN,FW: fixed forward or other Collar floor gas pr...,NaN,phillip.allen@enron.com,zimam@enron.com,2000-10-16 13:44:00+00:00,NaT,NaN,0
11,"Lucy,\n\n Here are the rentrolls:\n\n\n\n Open...",NaN,,NaN,phillip.allen@enron.com,stagecoachmama@hotmail.com,2000-10-13 13:45:00+00:00,NaT,NaN,0


In [37]:
result.shape

(344374, 10)

In [38]:
enron.shape

(15377, 13)

In [45]:
# Mirror enron's engineered columns onto result
result['DateSend'] = result['DateSend'].dt.tz_localize(None)
result['DateReply'] = result['DateReply'].dt.tz_localize(None)

result['Response_Within_30_Mins'] = (
    (result['DateReply'] - result['DateSend']) <= pd.Timedelta(minutes=30)
).astype(int)

result['HourSent'] = result['DateSend'].dt.hour
result['DaySent'] = result['DateSend'].dt.dayofweek
result['IsWeekend'] = result['DaySent'].isin([5, 6]).astype(int)

# Drop the extra column if it's still there
result = result.drop(columns=['Response'], errors='ignore')

# Align column order
result = result[enron.columns]

# Tag responded vs not responded
enron['Responded'] = 1
result['Responded'] = 0

# Join into one master dataset
master_enron = pd.concat([enron, result], ignore_index=True)

master_enron.head()

,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,Response_Within_30_Mins,HourSent,DaySent,IsWeekend,Responded
0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,0,22.0,0.0,0,1
1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,1,8.0,1.0,0,1
2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,0,12.0,1.0,0,1
3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,1,10.0,5.0,1,1
4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,1,15.0,0.0,0,1


In [60]:
master_enron.iloc[352001]['EmailSend']

"-----Original Message----- From: Ken Lay - Office of the Chairman Sent: Tuesday, November 13, 2001 5:18 PM To: DL-GA-all_enron_worldwide2 Subject: Change of Control Provisions As many of you know, I have a provision in my employment contract which provides for a payment of $20 million per year for the remaining term of my contract in the event of a change of control of Enron. The merger with Dynegy, or a similar transaction with any other company, would trigger this provision on closing. Assuming the merger with Dynegy is closed within 6-9 months, as we expect, this provision would entitle me to total payments of slightly more than $60 million. Many CEOs have change of control provisions in their employment contracts and mine has been in place since 1989. But given the current circumstances facing the company and our employees, I have been giving a lot of thought these last few days to what to do about this payment. Initially, I thought I would use part of the funds for a foundation f